# 06 Analysis

**Owner:** Part A each, Part B and C together

**What this notebook does:** answers the research question: how does democracy change before, during and after a conflict starts, and is it different for high and low arms importers?

**Reads:** `panel.csv`

**Writes:** `event_time.csv`, `event.csv`

This is a main notebook. It must run from top to bottom with Kernel → Restart and Run All.
Write simple code in small steps: one operation per line, print the result after each step.

## Setup

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

RAW = "../data/raw/"
CLEAN = "../data/clean/"
pd.set_option("display.max_columns", 50)

In [ ]:
panel = pd.read_csv(CLEAN + "panel.csv")
print(panel.shape)

---
# Part A: Descriptives

## Nicolas: conflict by regime type

Paste from sandbox_nicolas, Part C. One sentence of result below each table.

In [ ]:
# your code here


closed = panel[panel["regime_label"] == "closed autocracy"]

n_rows = len(closed)
n_conflict = len(closed[closed["conflict_active"] == 1])

share = n_conflict / n_rows
print("closed autocracy:", n_rows, "rows,", n_conflict, "in conflict, share", round(share, 3))

In [ ]:
# your code here


labels = ["closed autocracy", "electoral autocracy", "electoral democracy", "liberal democracy"]

conflict_share = {}

for label in labels:
    rows = panel[panel["regime_label"] == label]
    rows_conflict = rows[rows["conflict_active"] == 1]
    share = len(rows_conflict) / len(rows)
    conflict_share[label] = share

print(conflict_share)

print(panel.groupby("regime_label")["conflict_active"].mean())

## Yair: arms imports, democracy in war and peace

Paste from sandbox_yair, Part B. One sentence of result below each table.

In [ ]:
# your code here

In [ ]:
# your code here

In [ ]:
# your code here

---
# Part B: Event window (together)

The idea: we line up all conflicts as if they started in the same year, called `t = 0`.
Then we look at the average democracy index from 5 years before (`t = -5`) to 10 years after (`t = 10`).
Everything here uses dictionaries and loops, so you can check each step by printing.

**Step 1.** **Find the onsets.** An onset is a country-year with `conflict_active == 1` where the two years before had no conflict. First make a dictionary `status = {(iso3, year): conflict_active}` with one loop over `panel`. Then loop over `panel` again: a row is an onset if its `conflict_active` is 1 and `status.get((code, year - 1)) == 0` and `status.get((code, year - 2)) == 0`. Append `{"iso3": code, "onset_year": year}` to a list and make the dataframe `onsets`. Print how many onsets you found and the ones for Colombia and Ukraine.

In [ ]:
# your code here


status = {}

for i, row in panel.iterrows():
    key = (row["iso3"], row["year"])
    status[key] = row["conflict_active"]

print("country-years in status:", len(status))

In [ ]:
# your code here


onset_list = []

for i, row in panel.iterrows():
    code = row["iso3"]
    year = row["year"]
    if row["conflict_active"] == 1:
        if status.get((code, year - 1)) == 0 and status.get((code, year - 2)) == 0:
            onset_list.append({"iso3": code, "onset_year": year})

onsets = pd.DataFrame(onset_list)

print("number of onsets total list:", len(onsets))
print(onsets[onsets["iso3"] == "COL"])
print(onsets[onsets["iso3"] == "UKR"])

<details><summary><b>Hint</b> (open only if stuck)</summary>

`status.get(key)` returns `None` if the key does not exist. So years before 1950, and years right after a gap in the data,
can never count as "peaceful". That is what we want: a conflict already running in 1950 or 1951 cannot count as an onset.
Write that in the decisions table, and ask Justinas whether "two peaceful years" is a good definition.

</details>

**Step 2.** **Build the window.** Empty list `window`. Loop over `onsets`. For each onset, take the rows of that country with `year` between `onset_year - 5` and `onset_year + 10`. Loop over those rows and append a dictionary with `iso3`, `onset_year`, `year`, `t` (= `year - onset_year`), `democracy_index`, `arms_imports_tiv` and `max_intensity`. Make a dataframe `event`. Print the rows of one onset.

In [ ]:
# your code here

window = []

for i, onset in onsets.iterrows():
    code = onset["iso3"]
    onset_year = onset["onset_year"]
    country_rows = panel[panel["iso3"] == code]
    years_around = country_rows[country_rows["year"].between(onset_year - 5, onset_year + 10)]
    for j, row in years_around.iterrows():
        window.append({
            "iso3": code,
            "onset_year": onset_year,
            "year": row["year"],
            "t": row["year"] - onset_year,
            "democracy_index": row["democracy_index"],
            "arms_imports_tiv": row["arms_imports_tiv"],
            "max_intensity": row["max_intensity"],
        })

event = pd.DataFrame(window)
print("rows in the event:", len(event))

In [ ]:
# your code here

first = onsets.iloc[0]
print(event[(event["iso3"] == first["iso3"]) & (event["onset_year"] == first["onset_year"])])

<details><summary><b>Hint</b> (open only if stuck)</summary>

One country: `country_rows = panel[panel["iso3"] == code]`. The years around the onset: `country_rows[country_rows["year"].between(onset_year - 5, onset_year + 10)]`.

</details>

**Step 3.** **Change since the year before the conflict.** Make a dictionary `base = {}`. Loop over the rows of `event` with `t == -1`; if the democracy value is not missing (`not pd.isna(value)`), store `base[(iso3, onset_year)] = value`. Then loop over `event` again and build a list `changes`: if the key is in `base`, append `democracy_index - base[key]`, otherwise append `None`. Add it as the column `dem_change`. Print how many onsets have no base value. Why do we use the change and not the level? Write one sentence.

In [ ]:
# your code here


base = {}

year_before = event[event["t"] == -1]
for i, row in year_before.iterrows():
    value = row["democracy_index"]
    if not pd.isna(value):
        key = (row["iso3"], row["onset_year"])
        base[key] = value

changes = []
for i, row in event.iterrows():
    key = (row["iso3"], row["onset_year"])
    if key in base:
        changes.append(row["democracy_index"] - base[key])
    else:
        changes.append(None)

event["dem_change"] = changes

In [ ]:
# your code here


print("onsets:", len(onsets))
print("onsets with a base value:", len(base))
print("onsets without a base value:", len(onsets) - len(base))

**Step 4.** **High and low importers.** For each onset, compute the average `arms_imports_tiv` over `t = -5` to `t = -1`: loop over `event[event["t"] < 0]` and collect the values per onset in a dictionary of lists, then take the average of each list. Find the median of these averages: `pd.Series(list(averages.values())).median()` if your dictionary is called `averages`. Above the median = `"high importer"`, otherwise `"low importer"`. Store it in a dictionary and add a column `importer_group` to `event` with a loop. Print the median, the share of onsets with an average of 0, and how many onsets are in each group.

In [ ]:
# your code here

tiv_lists = {}

before = event[event["t"] < 0]
for i, row in before.iterrows():
    key = (row["iso3"], row["onset_year"])
    if key not in tiv_lists:
        tiv_lists[key] = []
    tiv_lists[key].append(row["arms_imports_tiv"])

averages = {}
for key in tiv_lists:
    values = tiv_lists[key]
    averages[key] = sum(values) / len(values)

median = pd.Series(list(averages.values())).median()
print("median imports before the onset: ", median)

n_zero = 0
for key in averages:
    if averages[key] == 0:
        n_zero = n_zero + 1
print(n_zero / len(averages))

In [ ]:
# your code here

<details><summary><b>Hint</b> (open only if stuck)</summary>

Many countries import nothing before a conflict, so the median may be 0. Then "above the median" simply means "imported anything".
That is fine, but write it in the decisions table ("the split is some imports vs none, groups are not equal in size").

</details>

**Step 5.** **Average per year.** One groupby with `agg`, as in the week 4 Disney notebook: `event_time = event.groupby(["t", "importer_group"], as_index=False).agg(dem_change=("dem_change", "mean"), n_countries=("dem_change", "count"))`. `count` counts the onsets that have a value at this `t` (write that in a comment). The columns must be `t`, `importer_group`, `dem_change`, `n_countries`.

In [ ]:
# your code here

group = {}
for key in averages:
    if averages[key] > median:
        group[key] = "high importer"
    else:
        group[key] = "low importer"

groups = []
for i, row in event.iterrows():
    key = (row["iso3"], row["onset_year"])
    groups.append(group.get(key))

event["importer_group"] = groups

print(pd.Series(list(group.values())).value_counts())
print("rows without group: ", event["importer_group"].isna().sum()) 

In [ ]:
event_time = event.groupby(["t", "importer_group"], as_index=False).agg(
    dem_change=("dem_change", "mean"),
    n_countries=("dem_change", "count"),
)
print(event_time)



print(event_time[event_time["t"].isin([-5, -1, 0, 5, 10])])

**Step 6.** Save `event_time.csv` with `index=False`. Also save `event` as `event.csv`; you need it in Part C.

In [ ]:
# your code here


event_time.to_csv(CLEAN + "event_time.csv", index=False)
event.to_csv(CLEAN + "event.csv", index=False)
print("saved event_time.csv and event.csv")

**Checkpoint.** Run this cell. OK means done. Do not change it.

In [ ]:
et = pd.read_csv(CLEAN + "event_time.csv")
assert list(et.columns) == ["t", "importer_group", "dem_change", "n_countries"]
assert et["t"].min() == -5 and et["t"].max() == 10
assert set(et["importer_group"].unique()) == {"high importer", "low importer"}
assert abs(et[et["t"] == -1]["dem_change"]).max() < 1e-9, "dem_change must be 0 at t = -1"
assert et["dem_change"].isna().sum() == 0
assert et["n_countries"].min() > 0
print(et[et["t"] == 0])
print("OK")

#in both groups of high or low importers, the democracy index rises a bit around and after the conflict starts, but a bit moer for high importer class

---
# Part C: Checks and findings (together)

**Step 1. Wars only.** Make a set of onset keys `(iso3, onset_year)` where `max_intensity == 2` in any year from `t = 0` to `t = 5`.
Filter `event` to those keys with a loop, keep the importer groups from the full sample, and redo only Step 5. Does the picture stay the same?

**Step 2. Group sizes.** How many onsets are in each group? Fewer than about 20 makes the lines unreliable. Say so.

**Step 3. Changing samples.** Plot `n_countries` by `t`. Then redo Step 5 with only onsets between 1955 and 2015 (full window),
and with only the first onset per country (onsets less than 10 years apart overlap). Do the lines change? Write your choice in the decisions table.

In [ ]:
# your code here

war_keys = set()

first_years = event[(event["t"] >= 0) & (event["t"] <= 5)]
for i, row in first_years.iterrows():
    if row["max_intensity"] == 2:
        war_keys.add((row["iso3"], row["onset_year"]))

print("cases where after t = 0 until t < 5 the max intensity is reached", len(war_keys))

keep = []
for i, row in event.iterrows():
    key = (row["iso3"], row["onset_year"])
    keep.append(key in war_keys)

event_war = event[keep]

war_time = event_war.groupby(["t", "importer_group"], as_index=False).agg(
    dem_change=("dem_change", "mean"),
    n_countries=("dem_change", "count"),
)
print(war_time[war_time["t"].isin([0, 5, 10])])

In [ ]:
# your code here

print("all onsets: ")
print(pd.Series(list(group.values())).value_counts())

war_groups = []
for key in war_keys:
    war_groups.append(group[key])

print("wars only")
print(pd.Series(war_groups).value_counts())

# All groups have more than 20 onsets (smallest: 45 low importers among wars), so the lines are not based on too few cases.

In [ ]:
# your code here


fig, ax = plt.subplots(figsize=(8, 4))
for name in ["high importer", "low importer"]:
    rows = event_time[event_time["importer_group"] == name]
    ax.plot(rows["t"], rows["n_countries"], label=name)
ax.set_xlabel("years since conflict start (t)")
ax.set_ylabel("onsets with a value")
ax.set_title("Number of onsets per year in the window")
ax.legend()
plt.show()

full = event[(event["onset_year"] >= 1955) & (event["onset_year"] <= 2015)]
full_time = full.groupby(["t", "importer_group"], as_index=False).agg(
    dem_change=("dem_change", "mean"),
    n_countries=("dem_change", "count"),
)
print("full window only (onsets 1955 to 2015):")
print(full_time[full_time["t"].isin([0, 5, 10])])

first_onset = {}
for i, row in onsets.iterrows():
    code = row["iso3"]
    if code not in first_onset:
        first_onset[code] = row["onset_year"]
    elif row["onset_year"] < first_onset[code]:
        first_onset[code] = row["onset_year"]

keep = []
for i, row in event.iterrows():
    keep.append(first_onset[row["iso3"]] == row["onset_year"])

event_first = event[keep]

first_time = event_first.groupby(["t", "importer_group"], as_index=False).agg(
    dem_change=("dem_change", "mean"),
    n_countries=("dem_change", "count"),
)
print("first onset per country only:")
print(first_time[first_time["t"].isin([0, 5, 10])])

## Findings

Write three findings, one sentence each, with a number. These go into the app, the slides and the report.

1.#in both groups of high or low importers, the democracy index rises a bit around and after the conflict starts, but a bit moer for high importer class
2.democracy tends to rise in t = -5 until t = 0, so it seems the rise of democracy just further increases even when the war has started

3.

**What this cannot show:** (one sentence, for example that it shows a pattern, not a cause)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

CLEAN = "../data/clean/"
panel = pd.read_csv(CLEAN + "panel.csv")

# 1. find the conflict onsets (conflict after 2 years without conflict)
status = {}
for i, row in panel.iterrows():
    status[(row["iso3"], row["year"])] = row["conflict_active"]

onset_list = []
for i, row in panel.iterrows():
    code = row["iso3"]
    year = row["year"]
    if row["conflict_active"] == 1:
        if status.get((code, year - 1)) == 0 and status.get((code, year - 2)) == 0:
            onset_list.append((code, year))

# 2. average arms imports in the 5 years before each onset
tiv = {}
for i, row in panel.iterrows():
    tiv[(row["iso3"], row["year"])] = row["arms_imports_tiv"]

averages = {}
for code, onset_year in onset_list:
    values = []
    for year in range(onset_year - 5, onset_year):
        if (code, year) in tiv:
            values.append(tiv[(code, year)])
    if len(values) > 0:
        averages[(code, onset_year)] = sum(values) / len(values)

# 3. split at the median
median = pd.Series(list(averages.values())).median()

low_log = []
high_log = []
for key in averages:
    log_value = np.log10(averages[key] + 1)
    if averages[key] > median:
        high_log.append(log_value)
    else:
        low_log.append(log_value)

# 4. chart
log_median = np.log10(median + 1)
bins_low = np.linspace(0, log_median, 13)
bins_high = np.linspace(log_median, 3.8, 18)
bins = np.concatenate([bins_low, bins_high[1:]])

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(low_log, bins=bins, color="#8FB3D9", label="low importer (" + str(len(low_log)) + " onsets)")
ax.hist(high_log, bins=bins, color="#1D4E89", label="high importer (" + str(len(high_log)) + " onsets)")
ax.axvline(log_median, color="#C0392B", linestyle="--", label="cutoff = " + str(round(median, 1)) + " TIV per year")
ax.text(log_median - 0.05, 38, "low: 0 to " + str(round(median, 1)) + " TIV", ha="right", color="#C0392B")
ax.text(log_median + 0.05, 38, "high: above " + str(round(median, 1)) + " TIV", ha="left", color="#C0392B")
ax.set_xticks([0, 1, 2, 3])
ax.set_xticklabels(["0", "10", "100", "1,000"])
ax.set_xlabel("average arms imports in the 5 years before onset (TIV per year, log scale)")
ax.set_ylabel("number of onsets")
ax.set_title("Arms imports before conflict onset: low and high importers")
ax.legend(loc="upper right")
plt.show()

## Decisions in this notebook

Write every choice you made here (and copy it into the decisions log in `CONVENTIONS.md`). You need these for the report.

| Decision | Why |
| --- | --- |
|  |  |